<a href="https://colab.research.google.com/github/akgupta63/CODSOFT_TASK1/blob/main/task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# Let's inspect the files to understand the dataset and task
files = ['/content/description.txt', '/content/train_data.txt', '/content/test_data.txt', '/content/test_data_solution.txt']

for f_path in files:
    if os.path.exists(f_path):
        print(f"=== {f_path} (First 10 lines) ===")
        with open(f_path, 'r', encoding='utf-8', errors='ignore') as f:
            for i in range(10):
                line = f.readline()
                if not line:
                    break
                print(line.strip())
        print("\n")
    else:
        print(f"{f_path} does not exist.\n")

=== /content/description.txt (First 10 lines) ===
Train data:
ID ::: TITLE ::: GENRE ::: DESCRIPTION
ID ::: TITLE ::: GENRE ::: DESCRIPTION
ID ::: TITLE ::: GENRE ::: DESCRIPTION
ID ::: TITLE ::: GENRE ::: DESCRIPTION

Test data:
ID ::: TITLE ::: DESCRIPTION
ID ::: TITLE ::: DESCRIPTION
ID ::: TITLE ::: DESCRIPTION


=== /content/train_data.txt (First 10 lines) ===
1 ::: Oscar et la dame rose (2009) ::: drama ::: Listening in to a conversation between his doctor and parents, 10-year-old Oscar learns what nobody has the courage to tell him. He only has a few weeks to live. Furious, he refuses to speak to anyone except straight-talking Rose, the lady in pink he meets on the hospital stairs. As Christmas approaches, Rose uses her fantastical experiences as a professional wrestler, her imagination, wit and charm to allow Oscar to live life and love to the full, in the company of his friends Pop Corn, Einstein, Bacon and childhood sweetheart Peggy Blue.
2 ::: Cupid (1997) ::: thriller ::: A

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("1. Loading datasets...")

# Helper function to read the custom formatted files
def load_data(file_path, has_genre=True):
    data = []
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            parts = line.strip().split(' ::: ')
            if len(parts) >= (4 if has_genre else 3):
                val_id = parts[0]
                title = parts[1]
                if has_genre:
                    genre = parts[2]
                    description = ' '.join(parts[3:])
                    data.append((val_id, title, genre, description))
                else:
                    description = ' '.join(parts[2:])
                    data.append((val_id, title, description))

    if has_genre:
        return pd.DataFrame(data, columns=['ID', 'TITLE', 'GENRE', 'DESCRIPTION'])
    else:
        return pd.DataFrame(data, columns=['ID', 'TITLE', 'DESCRIPTION'])

# Load files
train_df = load_data('/content/train_data.txt', has_genre=True)
test_df = load_data('/content/test_data.txt', has_genre=False)
test_sol_df = load_data('/content/test_data_solution.txt', has_genre=True)

print(f"Loaded Train Size: {train_df.shape[0]}")
print(f"Loaded Test Size: {test_df.shape[0]}")

print("\n2. Vectorizing text data (TF-IDF)...")
# Use TF-IDF with English stop words and sublinear TF scaling to boost performance
tfidf = TfidfVectorizer(max_features=25000, stop_words='english', sublinear_tf=True, ngram_range=(1, 2))
X_train = tfidf.fit_transform(train_df['DESCRIPTION'])
Y_train = train_df['GENRE']

X_test = tfidf.transform(test_df['DESCRIPTION'])
Y_test = test_sol_df['GENRE']

print("\n3. Training Logistic Regression model...")
# Use multinomial Logistic Regression with C=2.0 for balanced regularization
model = LogisticRegression(max_iter=500, C=2.0, solver='saga', n_jobs=-1)
model.fit(X_train, Y_train)

print("\n4. Evaluating model performance...")
Y_pred = model.predict(X_test)

accuracy = accuracy_score(Y_test, Y_pred)
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n")
print(classification_report(Y_test, Y_pred, zero_division=0))

1. Loading datasets...
Loaded Train Size: 54214
Loaded Test Size: 54200

2. Vectorizing text data (TF-IDF)...

3. Training Logistic Regression model...

4. Evaluating model performance...
Test Accuracy: 0.6037

Classification Report:

              precision    recall  f1-score   support

      action       0.50      0.33      0.40      1314
       adult       0.69      0.31      0.42       590
   adventure       0.63      0.20      0.31       775
   animation       0.52      0.08      0.14       498
   biography       0.00      0.00      0.00       264
      comedy       0.56      0.61      0.58      7446
       crime       0.32      0.04      0.08       505
 documentary       0.69      0.86      0.77     13096
       drama       0.56      0.78      0.65     13612
      family       0.52      0.12      0.19       783
     fantasy       0.44      0.06      0.10       322
   game-show       0.89      0.53      0.67       193
     history       0.00      0.00      0.00       243
      ho

# Movie Genre Classification - Internship Project

## Project Overview
This project aims to build an automated machine learning classifier that predicts the genre of a movie based on its textual description. Using natural language processing (NLP) techniques, this system classifies movies into various genres such as drama, documentary, comedy, and horror.

## Dataset
The dataset consists of three main files containing movie metadata:
* **`train_data.txt`**: Training records with structure `ID ::: TITLE ::: GENRE ::: DESCRIPTION` (54,214 records).
* **`test_data.txt`**: Unlabeled test records with structure `ID ::: TITLE ::: DESCRIPTION` (54,200 records).
* **`test_data_solution.txt`**: Test set ground truth solution with structure `ID ::: TITLE ::: GENRE ::: DESCRIPTION` (54,200 records).

## Methodology
1. **Data Parsing**: Safely extracted and parsed the custom ` ::: ` delimited records into pandas DataFrames.
2. **Feature Extraction**: Converted movie descriptions into numeric features using **TF-IDF Vectorization** (Term Frequency-Inverse Document Frequency):
   * Configured with English stop-words filtering.
   * Extracting both unigrams and bigrams (`ngram_range=(1, 2)`).
   * Limited to the top 25,000 features with sublinear TF scaling applied to minimize the impact of repetitive words.
3. **Model Selection**: Trained a multinomial **Logistic Regression** model optimized with the parallel `saga` solver to handle large-scale multi-class text categorization.

## Results & Performance
* **Overall Accuracy**: **60.37%**
* **Top Performing Genres** (by F1-Score):
  * **Western**: 0.84
  * **Documentary**: 0.77
  * **Game-Show**: 0.67
  * **Drama**: 0.65

## Key Learnings & Takeaways
* Engineered robust NLP preprocessing and feature scaling pipelines suited for highly imbalanced multi-class classification.
* Achieved optimized, fast, and multi-threaded training using SAGA solver optimizations on high-dimensional text matrices.